In [ ]:
import random
import time
import re
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from konlpy.tag import Okt

# ==========================================
# 1. 형태소 분석 및 불용어 정제 클래스
# ==========================================
class WatchaDataMiner:
    def __init__(self):
        # KoNLPy Okt 형태소 분석기 초기화
        self.okt = Okt() 
        
        # 워드클라우드 생성 시 의미 없는 단어 및 불용어 사전 정의
        # 형태소 분석 시 기본형(어간)으로 변환되므로 '하다', '되다' 등의 원형을 등록합니다.
        self.stopwords = {
            # 기본 동사/형용사 어간 및 부사 배제
            '하다', '되다', '이다', '있다', '없다', '않다', '그렇다', '아니다', 
            '같다', '어떻다', '이렇다', '저렇다', '진짜', '너무', '정말', '그냥', 
            '이거', '저거', '그거', '많다', '좋다', '나오다', '보다', '가다', '오다',
            # 핵심 요구사항: 영화 제목 및 대형 어휘 제외 사전
            '케데헌', '골든', '케이팝데몬헌터스', '영화', '드라마', '케이팝',
            '작품', '관람', '평점', '리뷰', '감독', '배우', '관객', '생각', '느낌'
        }

    def clean_and_tokenize(self, text):
        """텍스트 1차 정제 및 형태소 분석 (명사, 형용사 추출)"""
        # 1차 정제: 한글과 공백만 남기고 특수문자, 이모지 제거
        text = re.sub(r'[^가-힣\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        
        if not text:
            return ""

        # 2차 정제: 형태소 분석 (stem=True 옵션으로 '재밌어서' -> '재밌다'로 원형 복원)
        pos_result = self.okt.pos(text, stem=True)
        
        valid_words = []
        for word, pos in pos_result:
            # 명사(Noun)와 형용사(Adjective)만 추출하여 의미 있는 단어 확보
            if pos in ['Noun', 'Adjective']:
                # '아', '오' 같은 1글자 음절 제외 및 불용어 사전 필터링
                if len(word) > 1 and word not in self.stopwords:
                    valid_words.append(word)
                    
        return " ".join(valid_words)


# ==========================================
# 2. 실시간 스크롤 및 가상화 대응 크롤링 함수
# ==========================================
def scrape_watcha_infinite(target_url, target_count=10000):
    chrome_options = Options()
    chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
    driver = webdriver.Chrome(options=chrome_options)
    
    miner = WatchaDataMiner() # 기존 형태소 정제 클래스 활용
    raw_comments_set = set()
    processed_data = []
    
    print(f"[정보] 왓챠피디아 수집을 시작합니다. 목표 수량: {target_count}개")
    driver.get(target_url)
    time.sleep(5)
    
    scroll_attempts = 0
    fail_count = 0  # 연속으로 데이터가 안 늘어나는 횟수 체크
    
    while len(raw_comments_set) < target_count:
        # 1. 화면에 존재하는 코멘트 요소 수집
        list_items = driver.find_elements(By.CSS_SELECTOR, "ul._commentsUl_1j2s5_6 > li")
        
        new_elements_found = 0
        for item in list_items:
            try:
                raw_text = item.text.strip()
                if not raw_text: continue
                
                lines = [line.strip() for line in raw_text.split('\n') if line.strip()]
                if not lines: continue
                main_text = max(lines, key=len)
                
                if main_text and main_text not in raw_comments_set:
                    raw_comments_set.add(main_text)
                    cleaned_text = miner.clean_and_tokenize(main_text)
                    if cleaned_text:
                        processed_data.append({'raw_text': main_text, 'cleaned_words': cleaned_text})
                        new_elements_found += 1
            except:
                continue
                
        # 2. 봇 탐지 원천 차단: 랜덤 인간형 딜레이 (3.5초 ~ 6.5초 사이 유동적 대기)
        delay = random.uniform(3.5, 6.5)
        
        # 3. 방해 요소 처리 (간혹 나타나는 팝업 창이나 버튼 클릭 유도 우회)
        try:
            # 혹시 무한 스크롤 중간에 '더보기' 버튼이 생겼다면 강제 클릭
            more_btn = driver.find_element(By.XPATH, "//button[contains(text(), '더보기')]")
            driver.execute_script("arguments[0].click();", more_btn)
            print("[안내] 더보기 버튼을 감지하여 클릭했습니다.")
            time.sleep(2)
        except:
            pass

        # 4. 화면 스크롤 하강
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(delay)
        
        scroll_attempts += 1
        
        # 5. 엄격한 정지 감지 디버깅
        if new_elements_found == 0:
            fail_count += 1
            # 7회 연속으로 스크롤을 내려도 새 데이터가 전혀 안 올라오면 한계 도달로 판단
            if fail_count >= 7: 
                print("\n[경고] 현재 페이지의 물리적 로딩 한계에 도달했습니다.")
                break
        else:
            fail_count = 0 # 새 데이터 찾으면 초기화
            
        if scroll_attempts % 5 == 0:
            print(f"[상태] 스크롤 {scroll_attempts}회 | 수집된 고유 리뷰: {len(raw_comments_set)}개 (대기 시간: {delay:.2f}초)")

    return pd.DataFrame(processed_data)

# ==========================================
# 실행 제어 타겟 URL 전략 수정
# ==========================================
if __name__ == "__main__":
    # [참고] recommended(추천순) 수집 시 한계가 있을 경우, recent(최신순) 경로를 활용하여 추가 수집할 수 있습니다.
    WATCHA_URL = "https://pedia.watcha.com/ko/contents/mW42oMq/comments?order=recommended"
    
    result_df = scrape_watcha_infinite(WATCHA_URL, target_count=10000)
    
    desktop_path = os.path.expanduser("~/Desktop/kpop_demon_hunters_watcha_infinite.csv")
    result_df.to_csv(desktop_path, index=False, encoding='utf-8-sig')
    print(f"[완료] 데이터 저장 완료. 최종 데이터 수: {len(result_df)}개")

In [ ]:
import os
import random
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from konlpy.tag import Okt

# ==========================================
# [재선언] 커널 리셋 대비용 형태소 분석 클래스
# ==========================================
class WatchaDataMiner:
    def __init__(self):
        self.okt = Okt() 
        self.stopwords = {
            '하다', '되다', '이다', '있다', '없다', '않다', '그렇다', '아니다', 
            '같다', '어떻다', '이렇다', '저렇다', '진짜', '너무', '정말', '그냥', 
            '이거', '저거', '그거', '많다', '좋다', '나오다', '보다', '가다', '오다',
            '케데헌', '골든', '케이팝데몬헌터스', '영화', '드라마', '케이팝',
            '작품', '관람', '평점', '리뷰', '감독', '배우', '관객', '생각', '느낌'
        }

    def clean_and_tokenize(self, text):
        import re
        text = re.sub(r'[^가-힣\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        if not text: return ""
        pos_result = self.okt.pos(text, stem=True)
        valid_words = []
        for word, pos in pos_result:
            if pos in ['Noun', 'Adjective']:
                if len(word) > 1 and word not in self.stopwords:
                    valid_words.append(word)
        return " ".join(valid_words)

# ==========================================
# 1. 파일 복원
# ==========================================
desktop_path = os.path.expanduser("~/Desktop/kpop_demon_hunters_watcha_infinite.csv")

if os.path.exists(desktop_path):
    resume_df = pd.read_csv(desktop_path)
    print(f"[정보] 기존 데이터 {len(resume_df)}개 로드 완료.")
else:
    print("[정보] 기존 데이터 파일이 없어 새로 시작합니다.")
    resume_df = pd.DataFrame(columns=['raw_text', 'cleaned_words'])

raw_comments_set = set(resume_df['raw_text'].dropna().tolist())
processed_data = resume_df.to_dict('records')

# ==========================================
# 2. 브라우저 재연결
# ==========================================
try:
    chrome_options = Options()
    chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
    driver = webdriver.Chrome(options=chrome_options)
    print(f"[정보] 활성화된 크롬 브라우저 연결 성공: [{driver.title}]")
except Exception as e:
    print(f"[오류] 크롬 연결 실패. 디버깅 크롬 창 활성화 여부를 확인하십시오. 에러내용: {e}")

miner = WatchaDataMiner()
target_count = 10000
scroll_attempts = 0
consecutive_zero_count = 0 

print(f"[정보] 현재 누적 수량: {len(raw_comments_set)}개 | 수집을 재개합니다.")
print("=" * 60)

# ==========================================
# 3. 무한 루프 수집 (와일드카드 선택자 적용)
# ==========================================
while len(raw_comments_set) < target_count:
    try:
        # [참고] 클래스명 변경에 대응하기 위해 '_commentsUl'이 포함된 ul 요소를 탐색합니다.
        list_items = driver.find_elements(By.CSS_SELECTOR, "ul[class*='_commentsUl'] > li")
        
        new_elements_found = 0
        for item in list_items:
            try:
                raw_text = item.text.strip()
                if not raw_text: continue
                
                lines = [line.strip() for line in raw_text.split('\n') if line.strip()]
                if not lines: continue
                main_text = max(lines, key=len)
                
                if main_text and main_text not in raw_comments_set:
                    raw_comments_set.add(main_text)
                    cleaned_text = miner.clean_and_tokenize(main_text)
                    if cleaned_text:
                        processed_data.append({'raw_text': main_text, 'cleaned_words': cleaned_text})
                        new_elements_found += 1
            except Exception:
                continue
                
        delay = random.uniform(4.0, 6.0)
        
        # 더보기 버튼 자동 클릭
        try:
            more_btn = driver.find_element(By.XPATH, "//button[contains(text(), '더보기')]")
            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(1.5)
        except:
            pass

        # 스크롤 내리기
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(delay)
        scroll_attempts += 1
        
        if new_elements_found == 0:
            consecutive_zero_count += 1
            # 15번이나 스크롤 했는데 안 나오면 진짜 락 걸렸거나 끝인 것임 -> 이때는 파일 저장 후 브레이크
            if consecutive_zero_count >= 15: 
                print("\n[안내] 더 이상 새로운 리뷰가 로드되지 않습니다. 수집된 데이터를 저장합니다.")
                break
        else:
            consecutive_zero_count = 0 
            
        if scroll_attempts % 5 == 0:
            print(f"[상태] 스크롤 {scroll_attempts}회 | 총 데이터: {len(raw_comments_set)} / {target_count}개")

    except Exception as e:
        print(f"\n[경고] 일시적 오류가 발생했습니다 ({e}). 3초 후 재시도합니다.")
        time.sleep(3)
        continue

# ==========================================
# 4. 저장
# ==========================================
result_df = pd.DataFrame(processed_data)
result_df = result_df.drop_duplicates(subset=['raw_text']).reset_index(drop=True)
result_df.to_csv(desktop_path, index=False, encoding='utf-8-sig')

print("=" * 60)
print(f"[완료] 데이터 파일 저장 완료. 최종 고유 리뷰 수: {len(result_df)}개")

In [ ]:
import os
import random
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from konlpy.tag import Okt

# ==========================================
# [재선언] 커널 리셋 대비용 형태소 분석 클래스
# ==========================================
class WatchaDataMiner:
    def __init__(self):
        self.okt = Okt() 
        self.stopwords = {
            '하다', '되다', '이다', '있다', '없다', '않다', '그렇다', '아니다', 
            '같다', '어떻다', '이렇다', '저렇다', '진짜', '너무', '정말', '그냥', 
            '이거', '저거', '그거', '많다', '좋다', '나오다', '보다', '가다', '오다',
            '케데헌', '골든', '케이팝데몬헌터스', '영화', '드라마', '케이팝',
            '작품', '관람', '평점', '리뷰', '감독', '배우', '관객', '생각', '느낌'
        }

    def clean_and_tokenize(self, text):
        import re
        text = re.sub(r'[^가-힣\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        if not text: return ""
        pos_result = self.okt.pos(text, stem=True)
        valid_words = []
        for word, pos in pos_result:
            if pos in ['Noun', 'Adjective']:
                if len(word) > 1 and word not in self.stopwords:
                    valid_words.append(word)
        return " ".join(valid_words)

# ==========================================
# 1. 파일 복원
# ==========================================
desktop_path = os.path.expanduser("~/Desktop/kpop_demon_hunters_watcha_infinite.csv")

if os.path.exists(desktop_path):
    resume_df = pd.read_csv(desktop_path)
    print(f"[정보] 기존 데이터 {len(resume_df)}개 로드 완료.")
else:
    print("[정보] 기존 데이터 파일이 없어 새로 시작합니다.")
    resume_df = pd.DataFrame(columns=['raw_text', 'cleaned_words'])

raw_comments_set = set(resume_df['raw_text'].dropna().tolist())
processed_data = resume_df.to_dict('records')

# ==========================================
# 2. 브라우저 재연결 & 진짜 왓챠 탭 조준 (핵심 수정)
# ==========================================
try:
    chrome_options = Options()
    chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
    driver = webdriver.Chrome(options=chrome_options)
    
    # [참고] 열려 있는 모든 브라우저 탭을 확인하여 왓챠피디아 탭으로 포커스를 전환합니다.
    tab_found = False
    for handle in driver.window_handles:
        driver.switch_to.window(handle)
        if "watcha" in driver.current_url:
            print(f"[정보] 왓챠피디아 코멘트 탭을 탐색하여 연결했습니다: [{driver.title}]")
            tab_found = True
            break
            
    if not tab_found:
        print("[경고] 크롬은 연결되었으나, 왓챠피디아 코멘트 페이지 탭을 찾지 못했습니다.")
except Exception as e:
    print(f"[오류] 크롬 연결 실패. 에러내용: {e}")

miner = WatchaDataMiner()
target_count = 10000
scroll_attempts = 0
consecutive_zero_count = 0 

print(f"[정보] 현재 누적 수량: {len(raw_comments_set)}개 | 수집을 재개합니다.")
print("=" * 60)

# ==========================================
# 3. 무한 루프 수집
# ==========================================
while len(raw_comments_set) < target_count:
    try:
        list_items = driver.find_elements(By.CSS_SELECTOR, "ul[class*='_commentsUl'] > li")
        
        new_elements_found = 0
        for item in list_items:
            try:
                raw_text = item.text.strip()
                if not raw_text: continue
                
                lines = [line.strip() for line in raw_text.split('\n') if line.strip()]
                if not lines: continue
                main_text = max(lines, key=len)
                
                if main_text and main_text not in raw_comments_set:
                    raw_comments_set.add(main_text)
                    cleaned_text = miner.clean_and_tokenize(main_text)
                    if cleaned_text:
                        processed_data.append({'raw_text': main_text, 'cleaned_words': cleaned_text})
                        new_elements_found += 1
            except Exception:
                continue
                
        delay = random.uniform(4.0, 6.0)
        
        # 더보기 버튼 우회
        try:
            more_btn = driver.find_element(By.XPATH, "//button[contains(text(), '더보기')]")
            driver.execute_script("arguments[0].click();", more_btn)
            time.sleep(1.5)
        except:
            pass

        # 스크롤 내리기
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(delay)
        scroll_attempts += 1
        
        if new_elements_found == 0:
            consecutive_zero_count += 1
            if consecutive_zero_count >= 15: 
                print("\n[안내] 정렬 구조상 새로운 리뷰가 추가되지 않습니다. 수집을 종료하고 데이터를 저장합니다.")
                break
        else:
            consecutive_zero_count = 0 
            
        if scroll_attempts % 5 == 0:
            print(f"[상태] 스크롤 {scroll_attempts}회 | 총 데이터: {len(raw_comments_set)} / {target_count}개")

    except Exception as e:
        print(f"\n[경고] 일시적 오류가 발생했습니다 ({e}). 3초 후 재시도합니다.")
        time.sleep(3)
        continue

# ==========================================
# 4. 파일 갱신
# ==========================================
result_df = pd.DataFrame(processed_data)
result_df = result_df.drop_duplicates(subset=['raw_text']).reset_index(drop=True)
result_df.to_csv(desktop_path, index=False, encoding='utf-8-sig')

print("=" * 60)
print(f"[완료] 데이터 파일 저장 완료. 최종 고유 리뷰 수: {len(result_df)}개")